# 🌱 Monster Breach — Seed Dirty Crystals

> Run this notebook **ONCE** after `arcade.install("monster-breach")`.

It populates **`MonsterBreach_LH`** with the **dirty crystal** input tables (one per
level) plus the matching **clean oracle** tables the Judge uses to validate the data
pipelines you build.

## Requirements
1. Attach **`MonsterBreach_LH`** as the default Lakehouse (📚 icon → *Add* → Existing Lakehouse).
2. Run all cells top → bottom. Runtime: ~1–2 minutes.

## Tables created (all Delta)

| Level | Dirty input | Clean oracle | Defects to defeat |
|---|---|---|---|
| 1 Null Bug | `dirty_crystals_lvl01` | `clean_expected_lvl01` | NULL ids + text types |
| 2 Duplicate Ghost | `dirty_crystals_lvl02` | `clean_expected_lvl02` | duplicates + stale batches |
| 3 Branch Mimic | `dirty_crystals_lvl03` | `clean_expected_lvl03` | corrupted status + grade F |
| 4 Loop Wraith | `dirty_crystals_lvl04` (+ `mine_thresholds`) | `clean_expected_lvl04` | under-weight, per-mine thresholds |
| 5 Corruption King (BOSS) | `dirty_crystals_boss` | `clean_expected_boss` | ALL defects |

## Step 1 — Setup

In [ ]:
import random
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F, types as T, Row

# Deterministic seed so oracle values are reproducible across runs.
random.seed(7777)

MINES    = ["Cobalt", "Quartz", "Obsidian", "Amber", "Verdite"]
GRADES   = ["A", "B", "C", "F"]
STATUSES = ["Valid", "Corrupted"]

def cid(n):
    return f"CR-{n:06d}"

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true")\
      .format("delta").saveAsTable(name)
    print(f"  ✓ {name}: {df.count():,} rows")

print("Seed parameters loaded.")

## Step 2 — Level 1: Null Bug (drop NULL ids + cast types)

Dirty: ~18% of crystals have a **NULL `CrystalId`**; `WeightText` is text and
`PurityText` is a percent string like `"85%"`.
Clean: drop NULL ids, cast `Weight` to double, parse `Purity` to a 0–1 double.

In [ ]:
rows = []
n = 1200
null_idx = set(random.sample(range(n), int(n * 0.18)))
for i in range(n):
    w = round(random.uniform(1.0, 50.0), 3)
    p = random.randint(50, 99)
    rows.append(Row(
        CrystalId  = None if i in null_idx else cid(i),
        Mine       = random.choice(MINES),
        WeightText = f"{w}",
        PurityText = f"{p}%",
    ))
df = spark.createDataFrame(rows)
save(df, "dirty_crystals_lvl01")
clean = (df
    .filter(F.col("CrystalId").isNotNull())
    .withColumn("Weight", F.col("WeightText").cast("double"))
    .withColumn("Purity", (F.regexp_replace("PurityText", "%", "").cast("double") / 100.0))
    .select("CrystalId", "Mine", "Weight", "Purity"))
save(clean, "clean_expected_lvl01")

## Step 3 — Level 2: Duplicate Ghost (distinct + latest batch)

Dirty: each crystal appears 1–3 times and rows span four `IngestDate` batches.
Clean: keep only the **latest** `IngestDate`, then full-row **distinct**.

In [ ]:
batch_dates = [date(2049, 1, d) for d in (5, 12, 19, 26)]
latest = max(batch_dates)
base = []
for i in range(900):
    base.append(Row(
        CrystalId  = cid(i),
        Mine       = random.choice(MINES),
        Weight     = round(random.uniform(1.0, 50.0), 3),
        Purity     = round(random.uniform(0.50, 0.99), 4),
        IngestDate = random.choice(batch_dates),
    ))
dirty = []
for r in base:
    for _ in range(random.randint(1, 3)):
        dirty.append(r)
random.shuffle(dirty)
df = spark.createDataFrame(dirty)
save(df, "dirty_crystals_lvl02")
clean = df.filter(F.col("IngestDate") == F.lit(latest)).dropDuplicates()
save(clean, "clean_expected_lvl02")

## Step 4 — Level 3: Branch Mimic (Status Valid + drop grade F)

Dirty: each row has a `Status` (`Valid`/`Corrupted`) and a `Grade` (`A`/`B`/`C`/`F`).
Clean: keep rows where `Status = 'Valid'` **and** `Grade <> 'F'`.

In [ ]:
rows = []
for i in range(1300):
    rows.append(Row(
        CrystalId = cid(i),
        Mine      = random.choice(MINES),
        Weight    = round(random.uniform(1.0, 50.0), 3),
        Purity    = round(random.uniform(0.50, 0.99), 4),
        Status    = random.choices(STATUSES, weights=[7, 3])[0],
        Grade     = random.choices(GRADES, weights=[3, 3, 2, 2])[0],
    ))
df = spark.createDataFrame(rows)
save(df, "dirty_crystals_lvl03")
clean = df.filter((F.col("Status") == "Valid") & (F.col("Grade") != "F"))
save(clean, "clean_expected_lvl03")

## Step 5 — Level 4: Loop Wraith (per-mine weight thresholds)

Dirty: crystals tagged by `Mine`; the scales are corrupted with negative and
**under-weight** readings. Each mine has its **own** minimum safe weight, published
in the `mine_thresholds` table.
Clean: for each mine, keep only rows whose `Weight` exceeds **that mine's** `MinWeight`
(a blanket `Weight > 0` is deliberately *not* enough).

In [ ]:
# Each mine has its OWN minimum safe weight. A crystal is valid only if its Weight
# exceeds the threshold of its own mine — so a blanket Weight > 0 filter is wrong.
MINE_MIN_WEIGHT = {"Cobalt": 5.0, "Quartz": 10.0, "Obsidian": 2.0, "Amber": 20.0, "Verdite": 15.0}
thresholds = spark.createDataFrame(
    [Row(Mine=m, MinWeight=float(t)) for m, t in MINE_MIN_WEIGHT.items()])
save(thresholds, "mine_thresholds")

rows = []
for i in range(1500):
    # range spans below every per-mine threshold (incl. negatives) up to 50
    w = round(random.uniform(-5.0, 50.0), 3)
    rows.append(Row(
        CrystalId = cid(i),
        Mine      = random.choice(MINES),
        Weight    = w,
        Purity    = round(random.uniform(0.50, 0.99), 4),
    ))
df = spark.createDataFrame(rows)
save(df, "dirty_crystals_lvl04")
clean = (df.join(F.broadcast(thresholds), "Mine")
           .filter(F.col("Weight") > F.col("MinWeight"))
           .select("CrystalId", "Mine", "Weight", "Purity"))
save(clean, "clean_expected_lvl04")

## Step 6 — BOSS: Corruption King (every defect at once)

Dirty: one table containing NULL ids, duplicates, stale `IngestDate` batches, corrupted
status, grade F and under-weight crystals — all together.
Clean: keep the **latest** `IngestDate`, then `CrystalId` not null **and** `Status='Valid'`
**and** `Grade<>'F'` **and** `Weight >` the mine's `MinWeight` (from `mine_thresholds`),
then full-row distinct.

In [ ]:
batch_dates = [date(2049, 3, d) for d in (4, 11, 18, 25)]
latest = max(batch_dates)
rows = []
n = 2000
null_idx = set(random.sample(range(n), int(n * 0.10)))
for i in range(n):
    rows.append(Row(
        CrystalId  = None if i in null_idx else cid(i),
        Mine       = random.choice(MINES),
        Weight     = round(random.uniform(-8.0, 50.0), 3),
        Purity     = round(random.uniform(0.50, 0.99), 4),
        Status     = random.choices(STATUSES, weights=[7, 3])[0],
        Grade      = random.choices(GRADES, weights=[3, 3, 2, 2])[0],
        IngestDate = random.choice(batch_dates),
    ))
# inject duplicates
dirty = list(rows)
for r in random.sample(rows, 300):
    dirty.append(r)
random.shuffle(dirty)
df = spark.createDataFrame(dirty)
save(df, "dirty_crystals_boss")
# per-mine weight thresholds (same table Level 4 uses) instead of a blanket Weight > 0
mt = spark.table("mine_thresholds")
clean = (df
    .filter(F.col("IngestDate") == F.lit(latest))
    .filter(F.col("CrystalId").isNotNull())
    .filter(F.col("Status") == "Valid")
    .filter(F.col("Grade") != "F")
    .join(F.broadcast(mt), "Mine")
    .filter(F.col("Weight") > F.col("MinWeight"))
    .select("CrystalId", "Mine", "Weight", "Purity", "Status", "Grade", "IngestDate")
    .dropDuplicates())
save(clean, "clean_expected_boss")

## ✅ Done

All dirty + oracle tables are seeded. Open **`MonsterBreach_Judge`** and read the
Level 1 briefing to begin the hunt.

In [ ]:
print("🐉 Crystals seeded. Open MonsterBreach_Judge next.")